In [1]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.preprocessing import MultiLabelBinarizer


In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5 

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
ds = load_dataset("Rami/multi-label-class-github-issues-text-classification")

train_df = ds['train'].to_pandas()
val_df = ds['valid'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,title,labels,bodyText
0,TPUs: crash using torch-xla nightly,"[bug, help wanted, won't fix, accelerator: tpu]",🐛 Bug\nIf I try to use torch-xla nightly with ...
1,Fix docs typo in starter files,[docs],"📚 Documentation\nFor typos and doc fixes, plea..."
2,Fix typo in starter files,[docs],"📚 Documentation\nFor typos and doc fixes, plea..."
3,on_*_batch_transfer hooks should include a dat...,"[feature, help wanted]",🚀 Feature\nSee title\nMotivation\nUsers might ...
4,Allow arbitrary val check intervals when using...,"[feature, help wanted]",🚀 Feature\nCurrently when using the max epochs...
...,...,...,...
1551,Logger emits exception when there's `None` in ...,"[bug, help wanted]","To Reproduce\nMy hparams:\n{\n\t'n': [8000],\n..."
1552,TypeError: __init__() got an unexpected keywor...,"[bug, help wanted]","🐛 Bug\nI followed the guide to ""use 16bit prec..."
1553,Simplification: Merge load_from_metrics and lo...,"[feature, help wanted, good first issue]",🚀 Feature\nThe two ways of loading a Lightning...
1554,GPT2-large on Colab TPU seems to time out,"[bug, help wanted]",🐛 Bug\nWhen training gpt2-large on a colab tpu...


In [4]:
allowed_categories = ["bug", "feature", "question", "won't fix", "docs"]

train_df = train_df[train_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test_df = test_df[test_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val_df = val_df[val_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]

train_df = train_df[train_df['labels'].apply(len) > 0]
test_df = test_df[test_df['labels'].apply(len) > 0]
val_df = val_df[val_df['labels'].apply(len) > 0]

In [5]:
train_df.rename(columns={'title': 'text'}, inplace=True)
test_df.rename(columns={'title': 'text'}, inplace=True)
val_df.rename(columns={'title': 'text'}, inplace=True)

train_df.drop(columns=['bodyText'], inplace=True)
test_df.drop(columns=['bodyText'], inplace=True)
val_df.drop(columns=['bodyText'], inplace=True)

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

train_df

,text,labels
0,Fix docs typo in starter files,[docs]
1,Fix typo in starter files,[docs]
2,Load models give different results from original,[question]
3,Pickle error and OOM when upgrading to 1.2.0,"[question, won't fix]"
4,val_check_interval equivalent for training los...,[won't fix]
...,...,...
410,How to implement pre-training?,[question]
411,Logging the current learning rate,[question]
412,Example of gradient accumulation documentation...,[docs]
413,Checkpooint Callback not called when training ...,[question]


In [6]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train_df['labels'])
val_labels_binarized = mlb.transform(val_df['labels'])
test_labels_binarized = mlb.transform(test_df['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train_df = pd.concat([train_df, train_labels_df], axis=1)
val_df = pd.concat([val_df, val_labels_df], axis=1)
test_df = pd.concat([test_df, test_labels_df], axis=1)

train_df = train_df.drop(columns=['labels'])
val_df = val_df.drop(columns=['labels'])
test_df = test_df.drop(columns=['labels'])

train_df

,text,bug,docs,feature,question,won't fix
0,Fix docs typo in starter files,0,1,0,0,0
1,Fix typo in starter files,0,1,0,0,0
2,Load models give different results from original,0,0,0,1,0
3,Pickle error and OOM when upgrading to 1.2.0,0,0,0,1,1
4,val_check_interval equivalent for training los...,0,0,0,0,1
...,...,...,...,...,...,...
410,How to implement pre-training?,0,0,0,1,0
411,Logging the current learning rate,0,0,0,1,0
412,Example of gradient accumulation documentation...,0,1,0,0,0
413,Checkpooint Callback not called when training ...,0,0,0,1,0


In [7]:
class MultiLabelClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        """
        Args:
            texts: List or array of text samples
            labels: 2D array of shape (num_samples, num_classes) with binary indicators (0 or 1)
            tokenizer: Pretrained tokenizer (e.g., DistilBertTokenizer)
            max_len: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels  # Shape: (num_samples, num_classes)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]  # Shape: (num_classes,)
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)  # Binary vector for multilabel
        }

In [8]:
class DistilBertForMultiLabelClassification(nn.Module):
    def __init__(self, num_classes):
        super(DistilBertForMultiLabelClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output logits for each class
    
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits for BCEWithLogitsLoss

In [9]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [10]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, scheduler=None, epochs=EPOCHS):
    best_val_loss = float('inf')
    start_train = perf_counter()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False)
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # BCEWithLogitsLoss
            train_loss += loss.item()
            
            # Compute binary predictions for each class
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()  # Shape: (batch_size, num_classes)
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            
            loss.backward()
            optimizer.step()
            progress_bar.set_postfix({'loss': loss.item()})
        
        if scheduler:
            scheduler.step()
            
        train_loss /= len(train_dataloader)
        train_true = np.array(train_true)  # Shape: (num_samples, num_classes)
        train_preds = np.array(train_preds)  # Shape: (num_samples, num_classes)
        
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        start_val = perf_counter()
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc="Validation", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_time = perf_counter() - start_val
        
        val_loss /= len(val_dataloader)
        val_true = np.array(val_true)  # Shape: (num_samples, num_classes)
        val_preds = np.array(val_preds)  # Shape: (num_samples, num_classes)
        
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}, Prec: {train_precisions}, Recall: {train_recalls}")
        print(f"Epoch {epoch + 1}/{epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}, Prec: {val_precisions}, Recall: {val_recalls}, Val Time: {val_time:.2f} sec")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'results/bert_multilabel2.pt')
            print("Model saved!")
    
    total_train_time = perf_counter() - start_train
    print(f"Total Training Time: {total_train_time:.2f} seconds")
    
    return train_acc, train_precisions, train_recalls, train_f1s, val_acc, val_precisions, val_recalls, val_f1s, total_train_time, val_time

In [11]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].cpu().numpy()  # Shape: (num_classes,)
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = (torch.sigmoid(output) > 0.5).float().cpu().numpy()[0]  # Shape: (num_classes,)
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)  # Shape: (num_samples, num_classes)
    true_labels = np.array(true_labels)  # Shape: (num_samples, num_classes)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [12]:
train_texts = train_df['text'].values
train_labels = train_df.drop(columns=['text']).values

val_texts = val_df['text'].values
val_labels = val_df.drop(columns=['text']).values

test_texts = test_df['text'].values
test_labels = test_df.drop(columns=['text']).values

tokenizer = DistilBertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

# Use MultiClassClassificationDataset instead of BinaryClassificationDataset
train_dataset = MultiLabelClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiLabelClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiLabelClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

seeds = [2,3,5]

num_classes = len(allowed_categories)

avg_train_acc = 0
avg_train_precs = np.zeros(num_classes)
avg_train_recalls = np.zeros(num_classes)
avg_train_f1s = np.zeros(num_classes)
avg_max_memory_usage_train = 0
avg_max_vram_usage_train = 0
avg_total_train_time = 0

avg_val_acc = 0
avg_val_precs = np.zeros(num_classes)
avg_val_recalls = np.zeros(num_classes)
avg_val_f1s = np.zeros(num_classes)
avg_total_val_time = 0

avg_test_acc = 0
avg_test_precs = np.zeros(num_classes)
avg_test_recalls = np.zeros(num_classes)
avg_test_f1s = np.zeros(num_classes)
avg_max_memory_usage_test = 0
avg_max_vram_usage_test = 0
avg_total_test_time = 0

for seed in seeds:
    torch.manual_seed(seed)
    model = DistilBertForMultiLabelClassification(num_classes)
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()  # For multi-label

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion), {'epochs': EPOCHS}),
        max_usage=True,
        retval=True
    )

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s,
     total_train_time, val_time) = retval

    model.load_state_dict(torch.load('results/bert_multilabel2.pt'))

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}),
        max_usage=True,
        retval=True
    )
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0

    predictions, true_labels = retval

    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    avg_train_acc += train_acc
    avg_train_precs += train_precisions
    avg_train_recalls += train_recalls
    avg_train_f1s += train_f1s
    avg_max_memory_usage_train += max_memory_usage_train
    avg_max_vram_usage_train += max_vram_usage_train
    avg_total_train_time += total_train_time

    avg_val_acc += val_acc
    avg_val_precs += val_precisions
    avg_val_recalls += val_recalls
    avg_val_f1s += val_f1s
    avg_total_val_time += val_time

    avg_test_acc += test_acc
    avg_test_precs += test_precisions
    avg_test_recalls += test_recalls
    avg_test_f1s += test_f1s
    avg_max_memory_usage_test += max_memory_usage_test
    avg_max_vram_usage_test += max_vram_usage_test
    avg_total_test_time += total_time_test

avg_train_acc /= len(seeds)
avg_train_precs /= len(seeds)
avg_train_recalls /= len(seeds)
avg_train_f1s /= len(seeds)
avg_max_memory_usage_train /= len(seeds)
avg_max_vram_usage_train /= len(seeds)
avg_total_train_time /= len(seeds)

avg_val_acc /= len(seeds)
avg_val_precs /= len(seeds)
avg_val_recalls /= len(seeds)
avg_val_f1s /= len(seeds)
avg_total_val_time /= len(seeds)

avg_test_acc /= len(seeds)
avg_test_precs /= len(seeds)
avg_test_recalls /= len(seeds)
avg_test_f1s /= len(seeds)
avg_max_memory_usage_test /= len(seeds)
avg_max_vram_usage_test /= len(seeds)
avg_total_test_time /= len(seeds)

avg_classification_time = avg_total_test_time / len(test_texts)

avg_classification_time

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/3 - Train Loss: 0.5880, Acc: 0.3181, F1: [0.16666667 0.04878049 0.         0.76751592 0.10958904], Prec: [0.1969697  0.06451613 0.         0.64784946 0.13793103], Recall: [0.14444444 0.03921569 0.         0.94140625 0.09090909]
Epoch 1/3 - Val Loss: 0.4876, Acc: 0.4652, F1: [0.         0.         0.         0.78571429 0.        ], Prec: [0.         0.         0.         0.64705882 0.        ], Recall: [0. 0. 0. 1. 0.], Val Time: 0.37 sec
Model saved!


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2/3 - Train Loss: 0.4547, Acc: 0.4458, F1: [0.         0.         0.         0.76390977 0.        ], Prec: [0.         0.         0.         0.62102689 0.        ], Recall: [0.        0.        0.        0.9921875 0.       ]
Epoch 2/3 - Val Loss: 0.4439, Acc: 0.4439, F1: [0.         0.         0.         0.79037801 0.        ], Prec: [0.         0.         0.         0.67647059 0.        ], Recall: [0.         0.         0.         0.95041322 0.        ], Val Time: 0.44 sec
Model saved!


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Epoch 3/3 - Train Loss: 0.4261, Acc: 0.4289, F1: [0.         0.         0.         0.81260365 0.        ], Prec: [0.         0.         0.         0.70605187 0.        ], Recall: [0.         0.         0.         0.95703125 0.        ]
Epoch 3/3 - Val Loss: 0.4320, Acc: 0.3797, F1: [0.         0.         0.         0.75294118 0.        ], Prec: [0.         0.         0.         0.71641791 0.        ], Recall: [0.         0.         0.         0.79338843 0.        ], Val Time: 0.54 sec
Model saved!
Total Training Time: 16.10 seconds


C:\Users\Rafael\AppData\Local\Temp\ipykernel_18192\12187452.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multilabel2.p

Test Time: 1.18 seconds
Test Metrics:
Accuracy: 0.395
F1s: [0.         0.         0.         0.83088235 0.        ]
Precisions: [0.         0.         0.         0.77931034 0.        ]
Recalls: [0.         0.         0.         0.88976378 0.        ]



c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0.008815460500045447

In [13]:
# save results to txt
with open("results/bert_multilabel2.txt", "w") as f:
    f.write(f"Average Train Accuracy: {avg_train_acc}\n")
    f.write(f"Average Train Precisions: {avg_train_precs}\n")
    f.write(f"Average Train Recalls: {avg_train_recalls}\n")
    f.write(f"Average Train F1s: {avg_train_f1s}\n")
    f.write(f"Average Max Memory Usage Train: {avg_max_memory_usage_train}\n")
    f.write(f"Average Max VRAM Usage Train: {avg_max_vram_usage_train}\n")
    f.write(f"Average Total Train Time: {avg_total_train_time}\n")
    f.write("\n")
    f.write(f"Average Val Accuracy: {avg_val_acc}\n")
    f.write(f"Average Val Precisions: {avg_val_precs}\n")
    f.write(f"Average Val Recalls: {avg_val_recalls}\n")
    f.write(f"Average Val F1s: {avg_val_f1s}\n")
    f.write(f"Average Total Val Time: {avg_total_val_time}\n")
    f.write("\n")
    f.write(f"Average Test Accuracy: {avg_test_acc}\n")
    f.write(f"Average Test Precisions: {avg_test_precs}\n")
    f.write(f"Average Test Recalls: {avg_test_recalls}\n")
    f.write(f"Average Test F1s: {avg_test_f1s}\n")
    f.write(f"Average Max Memory Usage Test: {avg_max_memory_usage_test}\n")
    f.write(f"Average Max VRAM Usage Test: {avg_max_vram_usage_test}\n")
    f.write(f"Average Total Test Time: {avg_total_test_time}\n")
    f.write("\n")
    f.write(f"Average Classification Time: {avg_classification_time}\n")
    f.write(f"Lines classified {len(test_texts)}\n")

    f.close()